# Decoding odor, walk, and vision from LFADS factors — kcEXP00H

This notebook is the kcEXP00H analogue of the velocity-decoding section in
`tutorials/multisession/3_evaluation_KYC.ipynb`. Instead of decoding a *continuous*
behavioral variable (hand velocity) with ridge regression, we decode three **binary,
per-timepoint** stimulus channels — **odor**, **forward-walk (fw)**, and **vision** —
with logistic regression.

For each channel we ask two questions, exactly mirroring the Rouse tutorial:

- **Within-fly**: is the stimulus readable from this fly's LFADS factors?
- **Across-fly**: train on the other flies and test on a held-out fly. If this works, LFADS
  found a **shared, fly-invariant latent code** for the stimulus across whole-brain region
  sets that don't even have matching regions — the real multisession payoff.

We also plot **time-resolved** decoding (one decoder per frame) to see *when* the latents
start carrying each stimulus, and compare the predictive factor directions across the three
channels.

> **What goes into the decoder:** X = LFADS factors `(trials, time, n_factors)` flattened to
> `(trials*time, n_factors)` — every timepoint is one sample. y = the binary stimulus state at
> that timepoint. Model = `LogisticRegression` (classification), metric = **ROC-AUC** (odor and
> vision are class-imbalanced, so raw accuracy is misleading).

## Parameters

In [ ]:
import re
import sys
from glob import glob
from pathlib import Path

import numpy as np
import h5py
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9

# --- Directory containing lfads_output_*.h5 (from run_posterior_sampling) ---
# Set this after training with 2_run_pbt_kcEXP00H.py, e.g. the run's best_model dir:
#   runs/kcEXP00H_multisession_<timestamp>/pbt/kcEXP00H_multisession/<tag>/best_model
H5_DIR = None   # e.g. Path('../runs/kcEXP00H_multisession_XXXX/.../best_model')

# --- Directory with the source xarray (.nc) files holding the stimulus regressors ---
XARRAY_DIR = (
    '/media/server/gklab/KarenCheng/DATA/LFM_current/'
    'kcEXP00H/kcEXP00H_PythonNotebooks/xarray_files/'
)
# Path to the project so `utils.Xarray_UtilFns` is importable (provides load_group_xarrays)
sys.path.insert(0, '/media/server/gklab/KarenCheng/DATA/LFM_current/'
                   'kcEXP00H/kcEXP00H_PythonNotebooks')

# Windowing — MUST match 1_data_prep_kcEXP00H.ipynb so targets align with the factors.
PRE_FRAMES = 25     # frames before odour onset
POST_FRAMES = 150   # frames after odour onset  -> window length 175

# Drop the experiment-start baseline trials (is_baseline==True; the user's trials 1 & 10).
# They are all-off AND confounded with experiment onset. The *randomized* xxx trials
# (condition=='xxx' & not baseline) are kept as legitimate all-off negatives.
EXCLUDE_BASELINE = True

assert H5_DIR is not None, "Set H5_DIR to the directory containing lfads_output_*.h5 files."
H5_DIR = Path(H5_DIR)

# The three stimulus channels: name -> (per-timepoint state coord, per-trial presence coord)
MODALITIES = {
    'odor':   ('odor_state_frame', 'odor_trial_bool'),
    'walk':   ('fw_state_frame',   'fw_trial_bool'),
    'vision': ('vis_state_frame',  'vis_trial_bool'),
}

## Load LFADS factors

Same `merge_train_valid` helper as the evaluation notebook. Two gotchas carried over:

- `train_inds` / `valid_inds` are stored as **float32** — cast to `int` before indexing.
- Merging restores the **original trial order** (0..17), which is exactly the order the
  stimulus regressors will be built in, so factors and targets line up trial-for-trial.

In [ ]:
def merge_train_valid(train_data, valid_data, train_inds, valid_inds):
    n_samples = len(train_data) + len(valid_data)
    merged = np.full((n_samples, *train_data.shape[1:]), np.nan)
    merged[train_inds] = train_data
    merged[valid_inds] = valid_data
    return merged

h5_paths = sorted(glob(str(H5_DIR / 'lfads_output_*.h5')))
assert h5_paths, f"No lfads_output_*.h5 in {H5_DIR}"
print(f"Found {len(h5_paths)} HDF5 files")

factors = {}        # h5_stem -> (n_trials, n_time, n_fac)  in original trial order
h5_stems = []
for path in h5_paths:
    stem = Path(path).stem.replace('lfads_output_', '')
    h5_stems.append(stem)
    with h5py.File(path) as f:
        ti = f['train_inds'][()].astype(int)
        vi = f['valid_inds'][()].astype(int)
        factors[stem] = merge_train_valid(
            f['train_factors'][()], f['valid_factors'][()], ti, vi,
        )
    print(f"  {stem}: factors {factors[stem].shape}")
n_fac = next(iter(factors.values())).shape[-1]
print(f"n_factors = {n_fac}")

## Build the decoding targets from the xarray

The stimulus regressors live in the source `.nc` files as coordinates:

- `odor_state_frame`, `fw_state_frame`, `vis_state_frame` — the within-trial **on/off timing**
  (0/1 along the shared 1050-frame `time` axis).
- `odor_trial_bool`, `fw_trial_bool`, `vis_trial_bool` — whether each **trial** had that channel.
- `condition` ∈ {`VOF,VOx,VxF,Vxx,xOF,xOx,xxF,xxx`} (V=vision, O=odor, F=fw; `x`=off).

The full per-trial, per-timepoint binary regressor is
`reg[trial, t] = trial_bool[trial] AND (state_frame[t] > 0)`. We then crop it with the **same
odor-onset window** used in data prep (`onset ± PRE/POST`), so it aligns with the factors. We
match each h5 file to its xarray by checking which `fly_id` appears in the h5 filename.

In [ ]:
from utils.Xarray_UtilFns import load_group_xarrays

ds_list = load_group_xarrays(XARRAY_DIR)
xa_by_flyid = {ds.attrs['fly_id']: ds for ds in ds_list}
print(f"Loaded {len(ds_list)} xarray datasets")

def find_xarray(h5_stem):
    # fly_id is the full filename stem and appears inside the h5 filename
    matches = [fid for fid in xa_by_flyid if fid in h5_stem]
    if len(matches) != 1:
        raise KeyError(f"{h5_stem}: matched {matches} (expected exactly 1)")
    return matches[0], xa_by_flyid[matches[0]]

targets = {}      # h5_stem -> {modality -> (n_trials, n_time) binary}
baseline = {}     # h5_stem -> (n_trials,) bool
conditions = {}   # h5_stem -> (n_trials,) str
for stem in h5_stems:
    fid, ds = find_xarray(stem)
    # Window is defined by ODOR onset (same as data prep), used for every channel.
    onset = int(np.median(ds.attrs['odor_framestarts_within_trial']))
    win = slice(onset - PRE_FRAMES, onset + POST_FRAMES)
    targets[stem] = {}
    for mod, (state_var, bool_var) in MODALITIES.items():
        state = np.asarray(ds[state_var].values).reshape(-1)        # (1050,)
        trial_b = np.asarray(ds[bool_var].values).reshape(-1)       # (n_trials,)
        full = (trial_b[:, None].astype(float) * (state[None, :] > 0)).astype(int)
        targets[stem][mod] = full[:, win]                           # (n_trials, 175)
    baseline[stem] = np.asarray(ds['is_baseline'].values).reshape(-1).astype(bool)
    conditions[stem] = np.asarray(ds['condition'].values).reshape(-1).astype(str)
    # Alignment + length sanity checks
    nt = factors[stem].shape[0]
    for mod in MODALITIES:
        assert targets[stem][mod].shape == (nt, PRE_FRAMES + POST_FRAMES), \
            f"{stem}/{mod} shape {targets[stem][mod].shape}"
    print(f"  {stem}: {nt} trials | "
          + " ".join(f"{m}+={targets[stem][m].any(1).sum()}" for m in MODALITIES))

### Handle the baseline / `xxx` trials

`xxx` trials (all channels off) are valid **negatives** — you need stimulus-off trials to decode
stimulus presence. But the **experiment-start baseline** trials (`is_baseline==True`) are all-off
*and* confounded with experiment onset (different arousal/drift state), so by default we drop them
from both factors and targets (applied identically, preserving alignment). The randomized `xxx`
trials are kept.

In [ ]:
factors_kept = {}
targets_kept = {}
for stem in h5_stems:
    keep = ~baseline[stem] if EXCLUDE_BASELINE else np.ones(len(baseline[stem]), bool)
    factors_kept[stem] = factors[stem][keep]
    targets_kept[stem] = {m: targets[stem][m][keep] for m in MODALITIES}
    n_xxx = (conditions[stem][keep] == 'xxx').sum()
    print(f"  {stem}: kept {keep.sum()}/{len(keep)} trials "
          f"(dropped {int((~keep).sum())} baseline) | randomized xxx negatives: {n_xxx}")

## Decode: within-fly and across-fly

For each fly and channel:

- **within-fly** — split trials 70/30, train a logistic decoder on train-trial timepoints, score
  ROC-AUC on test-trial timepoints.
- **across-fly** — train on *all* timepoints from the other flies, test on the held-out fly.

We use `class_weight="balanced"` (odor/vision are imbalanced) and return `NaN` when a split has
only one class (so AUC is undefined).

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def _flat(X, y):
    return X.reshape(-1, X.shape[-1]), y.reshape(-1)


def fit_score(Xtr, ytr, Xte, yte):
    Xtr, ytr = _flat(Xtr, ytr)
    Xte, yte = _flat(Xte, yte)
    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        return np.nan
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, p)


def within_fly_split(n_trials, test_frac=0.3, seed=0):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n_trials)
    n_test = max(1, int(round(test_frac * n_trials)))
    return perm[n_test:], perm[:n_test]   # train_trials, test_trials


results = []
for mod in MODALITIES:
    for stem in h5_stems:
        Xf, yf = factors_kept[stem], targets_kept[stem][mod]
        # within-fly
        tr, te = within_fly_split(Xf.shape[0])
        auc_w = fit_score(Xf[tr], yf[tr], Xf[te], yf[te])
        results.append(dict(fly=stem, modality=mod,
                            holdout='within-fly', auc=auc_w))
        # across-fly
        Xtr = np.concatenate([factors_kept[s] for s in h5_stems if s != stem], 0)
        ytr = np.concatenate([targets_kept[s][mod] for s in h5_stems if s != stem], 0)
        auc_a = fit_score(Xtr, ytr, Xf, yf)
        results.append(dict(fly=stem, modality=mod,
                            holdout='across-fly', auc=auc_a))

results = pd.DataFrame(results)
results

### Plot decoding AUC by channel

In [ ]:
import seaborn as sns

g_order = ['within-fly', 'across-fly']
fig, axes = plt.subplots(1, len(MODALITIES), figsize=(4 * len(MODALITIES), 4), sharey=True)
for ax, mod in zip(axes, MODALITIES):
    sub = results[results.modality == mod]
    sns.barplot(sub, x='holdout', y='auc', order=g_order, ax=ax,
                errorbar='se', capsize=0.15)
    sns.stripplot(sub, x='holdout', y='auc', order=g_order, ax=ax,
                  color='k', size=3, alpha=0.6)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8)   # chance
    ax.set_ylim(0, 1)
    ax.set_title(mod)
    ax.set_xlabel('')
    ax.set_ylabel('Decoding ROC-AUC')
fig.suptitle('Stimulus decoding from LFADS factors (chance = 0.5)')
fig.tight_layout()

## Time-resolved decoding

Fit **one decoder per frame** (across-fly, leave-one-fly-out averaged) and plot AUC vs frame, with
the square-pulse stimulus overlaid. The odor pulse is OFF for the first `PRE_FRAMES` and ON
afterward, so the curve shows the **onset latency** and **persistence** of each stimulus in the
latent code.

> **Lag note:** for *sensory* signals the neural response *follows* stimulus onset (the brain lags
> the world) — the opposite of motor decoding, where neural activity leads behavior. We use lag 0
> here; the rising edge of each curve already reveals the representational latency.

In [ ]:
n_time = PRE_FRAMES + POST_FRAMES
frames = np.arange(n_time) - PRE_FRAMES   # 0 = odor onset

fig, axes = plt.subplots(1, len(MODALITIES), figsize=(4 * len(MODALITIES), 3.5), sharey=True)
for ax, mod in zip(axes, MODALITIES):
    auc_t = np.full(n_time, np.nan)
    for t in range(n_time):
        per_fly = []
        for stem in h5_stems:
            Xte = factors_kept[stem][:, t, :]
            yte = targets_kept[stem][mod][:, t]
            Xtr = np.concatenate(
                [factors_kept[s][:, t, :] for s in h5_stems if s != stem], 0)
            ytr = np.concatenate(
                [targets_kept[s][mod][:, t] for s in h5_stems if s != stem], 0)
            if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
                continue
            clf = LogisticRegression(class_weight='balanced', max_iter=1000)
            clf.fit(Xtr, ytr)
            per_fly.append(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]))
        if per_fly:
            auc_t[t] = np.mean(per_fly)
    ax.plot(frames, auc_t, lw=1.5)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8)
    ax.axvline(0, color='r', ls=':', lw=1, label='odor onset')
    ax.set_ylim(0, 1)
    ax.set_title(mod)
    ax.set_xlabel('Frame (0 = odor onset)')
    ax.set_ylabel('Across-fly ROC-AUC')
    ax.legend(loc='lower right', fontsize=7)
fig.suptitle('Time-resolved stimulus decoding')
fig.tight_layout()

## Do the same factors carry odor, walk, and vision?

Fit one logistic decoder per channel on all pooled timepoints (all flies) and compare the learned
**factor-weight directions** via cosine similarity. High similarity between two channels means they
ride on overlapping latent dimensions; low/negative means the model separates them into distinct
factor subspaces. This connects to the readout-weight mapping in
`3_region_contributions_kcEXP00H.ipynb` (which factors map onto which brain regions).

In [ ]:
coefs = {}
for mod in MODALITIES:
    X = np.concatenate([factors_kept[s].reshape(-1, n_fac) for s in h5_stems], 0)
    y = np.concatenate([targets_kept[s][mod].reshape(-1) for s in h5_stems], 0)
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(X, y)
    coefs[mod] = clf.coef_.reshape(-1)

mods = list(MODALITIES)
C = np.array([coefs[m] for m in mods])
C = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)
cos = C @ C.T

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cos, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(mods))); ax.set_xticklabels(mods)
ax.set_yticks(range(len(mods))); ax.set_yticklabels(mods)
for i in range(len(mods)):
    for j in range(len(mods)):
        ax.text(j, i, f"{cos[i, j]:.2f}", ha='center', va='center', fontsize=8)
ax.set_title('Cosine similarity of\ndecoder factor-weight directions')
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()

## Summary

- **Within-fly AUC > 0.5** means a stimulus is readable from this fly's LFADS factors.
- **Across-fly ≈ within-fly** means the latent code for that stimulus is **shared across flies** —
  the multisession claim, despite each fly having different region sets.
- The **time-resolved** curves show *when* each stimulus enters the latent code (onset latency)
  and how long it persists.
- The **cosine-similarity** matrix shows whether odor, walk, and vision share latent dimensions or
  occupy distinct factor subspaces.

To map the predictive factors back onto brain regions, continue in
`3_region_contributions_kcEXP00H.ipynb`.